# Four-orientation acrylic-backplate wall mount

Two printed parts: a display-side carrier and a wall-side plate with two integral
retaining studs. The original acrylic case stays in place. All dimensions are mm.

**Before printing or hanging:** the user confirmed threaded inserts behind
the four M3 mounting holes. The supplier STEP does not show the inserts, so
measure their thread depth and choose screws that engage securely without
bottoming out or reaching the PCB. See `README.md` for prototype limitations.

Run cells in order. Start OCP CAD Viewer in VS Code before running the viewer
cell. Each view replaces the previous one; edit the parameters and rerun all.

In [ ]:
import math
import os
from pathlib import Path

from build123d import (
    Align, Box, BuildPart, BuildSketch, Circle, Compound, Cone, Cylinder, GeomType,
    Locations, Mode, Plane, Pos, Rectangle, RectangleRounded, Rot,
    export_step, export_stl, extrude, import_step, mirror,
)
from OCP.BRepAdaptor import BRepAdaptor_Surface

CAD_DIR = Path.cwd() if Path.cwd().name.lower() == "cad" else Path.cwd() / "CAD"
assert (CAD_DIR / "reference" / "elecrow_v12_original_assembly.step").exists(), CAD_DIR

## Viewer

The headless switch exists for automated geometric checks; leave it unset for
normal interactive use.

In [ ]:
HEADLESS = os.environ.get("CAD_HEADLESS") == "1"
if not HEADLESS:
    import ocp_vscode
    ocp_vscode.find_and_set_port()
    if ocp_vscode.get_port() is None:
        raise RuntimeError("Start OCP CAD Viewer in VS Code, then rerun this cell.")
    ocp_vscode.show_clear()

def view(*objects, names=None, colors=None, alphas=None):
    if not HEADLESS:
        ocp_vscode.show(*objects, names=names, colors=colors, alphas=alphas,
                        axes=True, grid=True)

## Dimensions and coordinate system

The dimension parameters below use Elecrow's original STEP XY coordinates.
The assembled reference is **rigidly rotated**, not mirrored: in the viewer,
Z points from the wall toward the display and the supplier's engraved rear
face points toward the wall. The printable carrier and wall plate are reflected
in X to suit that physical orientation. The unequal mounting-hole position is
intentional.

The carrier's four bosses touch the engraved face; all other raised areas are
at least `ACRYLIC_GAP` away from its nominal flat surface. The complete supplier
reference is V1.2; verify it matches your V1.1 device.

In [ ]:
BACKPLATE_HOLES = (
    (3.5, 65.15),
    (128.5, 65.15),
    (128.5, 15.15),
    (17.334, 27.192),
)
BOARD_CENTER = (66.0, 40.15)
# Deliberately asymmetric: each rotation engages a different pair of slots.
STUD_OFFSETS = ((20.0, -8.0), (-2.0, -4.0))
STUD_CENTERS = tuple((BOARD_CENTER[0] + x, BOARD_CENTER[1] + y) for x, y in STUD_OFFSETS)

ACRYLIC_GAP = 1.0
CARRIER_THICKNESS = 6.55
CARRIER_BASE_THICKNESS = 3.0
CARRIER_RIM_WIDTH = 2.4
BACKPLATE_SIZE = (134.0, 84.7)
BACKPLATE_CORNER_RADIUS = 5.2
CARRIER_RIB_WIDTH = 6.0
ATTACHMENT_BOSS_DIAMETER = 8.5
M3_CLEARANCE = 3.4
M3_HEAD_COUNTERBORE = 6.6
M3_HEAD_RECESS = 2.2
# Original assembly STEP: approximately 6.1 mm head diameter, protruding
# around 1.01 mm from the nominal acrylic outer face.
CORNER_SCREWS = ((3.0, 3.0), (129.0, 3.0), (3.0, 77.3), (129.0, 77.3))
CORNER_HEAD_DIAMETER = 6.2
CORNER_HEAD_DIAMETER_CLEARANCE = 0.8
CORNER_HEAD_PROJECTION = 1.1
CORNER_HEAD_Z_CLEARANCE = 0.35
CORNER_HEAD_RELIEF_DEPTH = max(
    0.0, CORNER_HEAD_PROJECTION + CORNER_HEAD_Z_CLEARANCE - ACRYLIC_GAP
)

WALL_PLATE_THICKNESS = 3.2
WALL_PLATE_DIAMETER = 50.0
WALL_HOLE_OFFSETS = ((-1.5, -14.0), (1.5, 14.0))
WALL_SCREW_CLEARANCE = 4.8
WALL_HEAD_COUNTERBORE = 9.0
WALL_HEAD_RECESS = 2.0
WALL_TO_CARRIER_GAP = 0.1

STUD_DIAMETER = 4.0
STUD_HEIGHT = 3.35
STUD_HEAD_DIAMETER = 9.0
# The sloped underside of the head is 45 degrees for support-free FDM.
HEAD_TAPER_HEIGHT = (STUD_HEAD_DIAMETER - STUD_DIAMETER) / 2
HEAD_TOP_THICKNESS = 0.5
SLOT_WIDTH = 4.6
HEAD_CHANNEL_WIDTH = 9.6
SLOT_TRAVEL = 9.5
SLOT_PAD_WIDTH = 14.4
HEAD_CHANNEL_FLOOR = 3.0

assert CARRIER_BASE_THICKNESS == HEAD_CHANNEL_FLOOR
HEAD_TOP = WALL_PLATE_THICKNESS + STUD_HEIGHT + HEAD_TAPER_HEIGHT + HEAD_TOP_THICKNESS
CARRIER_BOTTOM = WALL_PLATE_THICKNESS + WALL_TO_CARRIER_GAP
ACRYLIC_Z = CARRIER_BOTTOM + CARRIER_THICKNESS + ACRYLIC_GAP
assert CARRIER_THICKNESS > HEAD_CHANNEL_FLOOR
assert WALL_PLATE_THICKNESS + STUD_HEIGHT > CARRIER_BOTTOM + HEAD_CHANNEL_FLOOR
assert HEAD_TOP < CARRIER_BOTTOM + CARRIER_THICKNESS
assert SLOT_WIDTH > STUD_DIAMETER and HEAD_CHANNEL_WIDTH > STUD_HEAD_DIAMETER
assert SLOT_PAD_WIDTH > HEAD_CHANNEL_WIDTH
def rotate_xy(point, angle):
    x, y = point
    if angle == 0: return (x, y)
    if angle == 90: return (-y, x)
    if angle == 180: return (-x, -y)
    if angle == 270: return (y, -x)
    raise ValueError(angle)

def add_center(point):
    return (BOARD_CENTER[0] + point[0], BOARD_CENTER[1] + point[1])

def printable_orientation(shape):
    # Transform the *new printed part* into the real assembly/print frame.
    # Do not reflect the supplier's actual, asymmetric device.
    return Pos(2 * BOARD_CENTER[0], 0, 0) * mirror(shape, about=Plane.YZ)

# Every orientation has two *separate*, one-way keyholes. Their wide entries
# face world-down after the display is rotated to that orientation.
TRACKS = []
for angle in (0, 90, 180, 270):
    for x, y in STUD_OFFSETS:
        start = add_center(rotate_xy((x, y), angle))
        entry = add_center(rotate_xy((x, y - SLOT_TRAVEL), angle))
        TRACKS.append((start, entry))
print(f"Nominal acrylic gap: {ACRYLIC_GAP:.2f} mm")
print(f"Wall to acrylic: {ACRYLIC_Z:.2f} mm")
print(f"Stud-head clearance below carrier top: {CARRIER_BOTTOM + CARRIER_THICKNESS - HEAD_TOP:.2f} mm")

## Supplier assembly and the four rear mounting points

Orange cylinders mark the *additional* holes, not the four corner assembly
screws. The user confirmed that the four additional holes have threaded
inserts; these are not represented in the supplier model. The supplier's front
plate, display, PCB and components, middle frame, engraved rear plate, and
original screws are all kept in their original relative positions. Use the
viewer tree to hide individual parts.

In [ ]:
supplier_assembly = import_step(CAD_DIR / "reference" / "elecrow_v12_original_assembly.step")
supplier_solids = list(supplier_assembly.solids())
assert len(supplier_solids) == 59, "Unexpected supplier STEP structure"
supplier_backplate = supplier_solids[50]

# Original STEP: text is on the nominal z=8 rear face; the PCB/display are
# at smaller z. Rotate the entire original assembly 180 degrees about Y.
# This is a proper rigid rotation, not a reflection of the original device.
supplier_to_print = (
    Pos(2 * BOARD_CENTER[0], 0, CARRIER_THICKNESS + ACRYLIC_GAP + 8.0)
    * Rot(0, 180, 0)
)
source_groups = [
    supplier_solids[50],                         # engraved rear acrylic
    supplier_solids[49],                         # middle frame
    Compound(children=supplier_solids[:47]),     # PCB and components
    supplier_solids[47],                         # LCD
    supplier_solids[48],                         # front acrylic
    Compound(children=supplier_solids[51:]),     # supplier fasteners
]
reference_names = ["Rear acrylic (engraved face)", "Middle frame", "PCB and components",
                   "Display", "Front acrylic", "Original screws"]
reference_colors = ["lightgray", "lightgray", "green", "black", "silver", "gray"]
reference_alphas = [0.45, 0.35, 1, 1, 0.4, 1]
reference_parts = [supplier_to_print * part for part in source_groups]
for part, name in zip(reference_parts, reference_names):
    part.label = name
acrylic_reference = reference_parts[0]
hole_markers = [
    Pos(2 * BOARD_CENTER[0] - x, y, CARRIER_THICKNESS + ACRYLIC_GAP - 0.5)
    * Cylinder(M3_CLEARANCE / 2, 4, align=(Align.CENTER, Align.CENTER, Align.MIN))
    for x, y in BACKPLATE_HOLES
]
view(*reference_parts, *hole_markers,
     names=reference_names + ["M3 hole 1", "M3 hole 2", "M3 hole 3", "M3 hole 4"],
     colors=reference_colors + ["orange"] * 4,
     alphas=reference_alphas + [1] * 4)

## 1. Carrier blank

The carrier is a continuous 3 mm sheet within the acrylic outline. A raised
perimeter and thicker pads for the keyholes give it a solid side profile
without making the entire sheet 6.55 mm thick. The access cutouts are made in
the next step. The 1 mm bosses are the only intended contact with the acrylic.
Print flat, with the bosses facing up.

In [ ]:
def rib(start, end, width=CARRIER_RIB_WIDTH):
    dx, dy = end[0] - start[0], end[1] - start[1]
    length = math.hypot(dx, dy)
    angle = math.degrees(math.atan2(dy, dx))
    with Locations(((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)):
        Rectangle(length, width, rotation=angle)
    with Locations(start, end):
        Circle(width / 2)

def beam(start, end, width=CARRIER_RIB_WIDTH):
    dx, dy = end[0] - start[0], end[1] - start[1]
    length = math.hypot(dx, dy)
    angle = math.degrees(math.atan2(dy, dx))
    with Locations(((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)):
        # Overlap joints instead of relying on tangential circular seams.
        Rectangle(length + 2.0, width, rotation=angle)

with BuildPart() as carrier_blank_builder:
    # Continuous printable base, matching the original acrylic silhouette.
    with BuildSketch():
        with Locations(BOARD_CENTER):
            RectangleRounded(*BACKPLATE_SIZE, BACKPLATE_CORNER_RADIUS)
    extrude(amount=CARRIER_BASE_THICKNESS)
    # A full-height perimeter closes the visible edges; access openings are
    # cut afterwards. The inner step reduces material and print time.
    with BuildSketch():
        with Locations(BOARD_CENTER):
            RectangleRounded(*BACKPLATE_SIZE, BACKPLATE_CORNER_RADIUS)
            RectangleRounded(
                BACKPLATE_SIZE[0] - 2 * CARRIER_RIM_WIDTH,
                BACKPLATE_SIZE[1] - 2 * CARRIER_RIM_WIDTH,
                BACKPLATE_CORNER_RADIUS - CARRIER_RIM_WIDTH,
                mode=Mode.SUBTRACT,
            )
    extrude(amount=CARRIER_THICKNESS)
    with BuildSketch():
        for start, entry in TRACKS:
            rib(start, entry, SLOT_PAD_WIDTH)
        beam((3.5, 65.15), (86.0, 65.15))
        # Route around the entire wireless-module/header opening.
        beam((86.0, 51.0), (128.5, 51.0))
        beam((128.5, 51.0), (128.5, 65.15))
        beam((30.0, 15.15), (128.5, 15.15))
        beam((17.334, 27.192), (30.0, 15.15))
        beam((46.0, 15.15), (46.0, 65.15))
        beam((86.0, 15.15), (86.0, 65.15))
        beam((46.0, 40.15), (86.0, 40.15))
        beam((66.0, 15.15), (66.0, 65.15))
        with Locations(*BACKPLATE_HOLES):
            Circle(ATTACHMENT_BOSS_DIAMETER / 2)
    extrude(amount=CARRIER_THICKNESS)
    with Locations(*[(x, y, CARRIER_THICKNESS) for x, y in BACKPLATE_HOLES]):
        Cylinder(ATTACHMENT_BOSS_DIAMETER / 2 - 0.15, ACRYLIC_GAP,
                 align=(Align.CENTER, Align.CENTER, Align.MIN))
carrier_blank = carrier_blank_builder.part.fix()
view(printable_orientation(carrier_blank), acrylic_reference,
     names=["Carrier blank", "Supplier rear acrylic"],
     colors=["royalblue", "lightgray"], alphas=[1, 0.25])

## 2. Access openings, keyholes and M3 clearance holes

Each orientation uses its own two **separate** one-way keyholes. The narrow
lower layer retains the wide stud heads; the wider recess above it is an open,
support-free print feature. When mounting, align both studs with the entries
**below** the resting positions, then lower the display. The wall plate stays
in the same position for all four display orientations.

The 6.6 mm counterbores accept compact M3 screw heads (check actual head
height and diameter). The corner screws are covered, not accessible: shallow
recesses on the acrylic-facing side provide clearance for their heads. Rear
ports, switch and wireless-module region remain accessible. The large left
notch leaves BOOT and RESET unobstructed. Check actual clearances.

In [ ]:
def all_tracks(width):
    for start, entry in TRACKS:
        rib(start, entry, width)

with BuildPart() as carrier_builder:
    from build123d import add
    add(carrier_blank)
    # Narrow one-way through-slots retain the heads.
    with BuildSketch(Plane.XY):
        all_tracks(SLOT_WIDTH)
        # The wide entry circles must pass through the *entire* carrier;
        # otherwise a stud head cannot enter the upper recess.
        for _, entry in TRACKS:
            with Locations(entry):
                Circle(HEAD_CHANNEL_WIDTH / 2)
    extrude(amount=CARRIER_THICKNESS + 0.1, mode=Mode.SUBTRACT)
    # Open larger channels accommodate the heads above the retaining layer.
    with BuildSketch(Plane.XY.offset(HEAD_CHANNEL_FLOOR)):
        all_tracks(HEAD_CHANNEL_WIDTH)
    extrude(amount=CARRIER_THICKNESS - HEAD_CHANNEL_FLOOR + 0.1, mode=Mode.SUBTRACT)
    for x, y in BACKPLATE_HOLES:
        with Locations((x, y, -0.1)):
            Cylinder(M3_CLEARANCE / 2, CARRIER_THICKNESS + ACRYLIC_GAP + 0.2,
                     align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
            Cylinder(M3_HEAD_COUNTERBORE / 2, M3_HEAD_RECESS + 0.1,
                     align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
    # Rectangles generously cover the openings on the acrylic reference;
    # the wireless and button apertures deliberately open to the outer edge.
    openings = {
        "UART0-OUT": (18.5, 70.0, 33.5, 78.7),
        "UART1-OUT": (38.5, 70.0, 53.5, 78.7),
        "I2C-OUT": (58.5, 70.0, 73.5, 78.7),
        "wireless module": (97.3, 55.3, 118.9, 83.5),
        "S1/S0": (69.95, 19.9, 76.95, 26.9),
        "speaker": (32.05, 2.6, 43.05, 12.05),
        "UART0-IN": (105.5, 0.5, 124.5, 10.4),
        "BOOT/RESET side notch": (-2.0, 18.27, 9.34, 40.78),
    }
    for x1, y1, x2, y2 in openings.values():
        with Locations(((x1 + x2) / 2, (y1 + y2) / 2, -0.1)):
            Box(x2 - x1, y2 - y1, CARRIER_THICKNESS + ACRYLIC_GAP + 0.2,
                align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
    # Hidden blind reliefs: keep the wall-facing sheet and outside corners
    # closed while providing clearance for the four factory screw heads.
    if CORNER_HEAD_RELIEF_DEPTH:
        with Locations(*[(x, y, CARRIER_THICKNESS - CORNER_HEAD_RELIEF_DEPTH)
                         for x, y in CORNER_SCREWS]):
            Cylinder((CORNER_HEAD_DIAMETER + CORNER_HEAD_DIAMETER_CLEARANCE) / 2,
                     CORNER_HEAD_RELIEF_DEPTH + 0.1,
                     align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
    # Small buzzer vent.
    with Locations((84.931, 75.845, -0.1)):
        Cylinder(1.6, CARRIER_THICKNESS + ACRYLIC_GAP + 0.2,
                 align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
carrier = carrier_builder.part.fix()
view(printable_orientation(carrier), acrylic_reference, names=["Carrier", "Supplier rear acrylic"],
     colors=["royalblue", "lightgray"], alphas=[1, 0.22])

## 3. Wall plate and studs

Two slightly offset wall screws are recessed in the wall plate, which prints with its flat
wall-facing face on the bed. The stud heads have 45° ramps and are printed
upright. Use wall fixings appropriate to your wall; the wall screw recess is
9 mm diameter and 2 mm deep. Mark the two holes using the actual plate; they
are not quite vertically aligned. Four openings preserve S1/S0 access in all
orientations.

In [ ]:
with BuildPart() as wall_builder:
    with BuildSketch():
        with Locations(BOARD_CENTER):
            Circle(WALL_PLATE_DIAMETER / 2)
    extrude(amount=WALL_PLATE_THICKNESS)
    with Locations(*[(x, y, WALL_PLATE_THICKNESS) for x, y in STUD_CENTERS]):
        Cylinder(STUD_DIAMETER / 2, STUD_HEIGHT,
                 align=(Align.CENTER, Align.CENTER, Align.MIN))
    with Locations(*[(x, y, WALL_PLATE_THICKNESS + STUD_HEIGHT) for x, y in STUD_CENTERS]):
        Cone(STUD_DIAMETER / 2, STUD_HEAD_DIAMETER / 2, HEAD_TAPER_HEIGHT,
             align=(Align.CENTER, Align.CENTER, Align.MIN))
    with Locations(*[(x, y, WALL_PLATE_THICKNESS + STUD_HEIGHT + HEAD_TAPER_HEIGHT)
                     for x, y in STUD_CENTERS]):
        Cylinder(STUD_HEAD_DIAMETER / 2, HEAD_TOP_THICKNESS,
                 align=(Align.CENTER, Align.CENTER, Align.MIN))
    wall_holes = [add_center(point) for point in WALL_HOLE_OFFSETS]
    with Locations(*[(x, y, -0.1) for x, y in wall_holes]):
        Cylinder(WALL_SCREW_CLEARANCE / 2, WALL_PLATE_THICKNESS + 0.2,
                 align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
    with Locations(*[(x, y, WALL_PLATE_THICKNESS - WALL_HEAD_RECESS) for x, y in wall_holes]):
        Cylinder(WALL_HEAD_COUNTERBORE / 2, WALL_HEAD_RECESS + 0.1,
                 align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
    switch_offset = (73.45 - BOARD_CENTER[0], 23.4 - BOARD_CENTER[1])
    for angle in (0, 90, 180, 270):
        x, y = add_center(rotate_xy(switch_offset, angle))
        with Locations((x, y, -0.1)):
            Box(7.0, 7.0, WALL_PLATE_THICKNESS + 0.2,
                align=(Align.CENTER, Align.CENTER, Align.MIN), mode=Mode.SUBTRACT)
wall_plate = wall_builder.part
view(printable_orientation(wall_plate), names=["Wall plate"], colors=["darkorange"])

## 4. Assembly and four orientations

These views show the **resting** position. The mount is gravity-retained, not
positively locked: lifting the display can release it. Do not use it where
accidental lifting, vibration, or a fall could injure anyone. The wall plate
stays fixed; rotate only the display/carrier. In each orientation the correct
pair of entries is below its resting positions.

In [ ]:
print_carrier = printable_orientation(carrier)
print_wall_plate = printable_orientation(wall_plate)
assembled_carrier = Pos(0, 0, CARRIER_BOTTOM) * print_carrier
assembled_original = [Pos(0, 0, CARRIER_BOTTOM) * part for part in reference_parts]
print_wall_plate.label = "Wall plate"
assembled_carrier.label = "Acrylic carrier"
for part, name in zip(assembled_original, reference_names):
    part.label = name
assembled_acrylic = assembled_original[0]

def orientation_view(angle):
    cx, cy = BOARD_CENTER
    turn = Pos(cx, cy, 0) * Rot(0, 0, angle) * Pos(-cx, -cy, 0)
    view(print_wall_plate, turn * assembled_carrier, *[turn * part for part in assembled_original],
         names=["Wall plate", "Carrier"] + reference_names,
         colors=["darkorange", "royalblue"] + reference_colors,
         alphas=[1, 1] + reference_alphas)
    print(f"{angle}°: enter from below in world coordinates, then lower to rest.")

In [ ]:
orientation_view(0)

In [ ]:
orientation_view(90)

In [ ]:
orientation_view(180)

In [ ]:
orientation_view(270)

## 5. Geometry checks

These checks test the physical layer order and actual supplier geometry,
the four hole centres and insertion paths, separate solids, and clearance
for the main supplier cutouts. They do not certify print strength, the fixing
hardware, actual button height, or wall safety.

In [ ]:
def volume_of(shape):
    if shape is None:
        return 0.0
    if isinstance(shape, (list, tuple)):
        return sum(volume_of(item) for item in shape)
    return shape.volume

assert len(carrier.solids()) == 1 and carrier.is_valid
assert len(wall_plate.solids()) == 1 and wall_plate.is_valid
# Check physical layer order: wall -> carrier -> engraved rear -> PCB -> LCD.
assert print_wall_plate.bounding_box().min.Z < assembled_carrier.bounding_box().min.Z
assert assembled_carrier.bounding_box().max.Z <= ACRYLIC_Z + 1e-6
pcb_body = Pos(0, 0, CARRIER_BOTTOM) * supplier_to_print * supplier_solids[0]
assert pcb_body.bounding_box().min.Z > assembled_original[0].bounding_box().max.Z
assert assembled_original[3].bounding_box().min.Z > assembled_original[0].bounding_box().max.Z
for x, y in BACKPLATE_HOLES:
    physical_x = 2 * BOARD_CENTER[0] - x
    assert not print_carrier.is_inside((physical_x, y, CARRIER_THICKNESS / 2))
    assert print_carrier.is_inside((physical_x - 3.8, y, CARRIER_THICKNESS + ACRYLIC_GAP / 2))
# The physical carrier must not collide with any original supplier part.
for name, part in zip(reference_names, reference_parts):
    overlap = volume_of(print_carrier.intersect(part))
    assert overlap < 1e-5, ("supplier clearance", name, overlap)
source_holes = []
for face in supplier_backplate.faces():
    if face.geom_type == GeomType.CYLINDER:
        cylinder = BRepAdaptor_Surface(face.wrapped).Cylinder()
        if abs(cylinder.Radius() - 1.6) < 0.001:
            point = cylinder.Axis().Location()
            source_holes.append((point.X(), point.Y()))
for x, y in BACKPLATE_HOLES:
    assert any(math.hypot(x - sx, y - sy) < 0.001 for sx, sy in source_holes)
    assert not carrier.is_inside((x, y, CARRIER_THICKNESS / 2))
    assert carrier.is_inside((x + 3.8, y, CARRIER_THICKNESS + ACRYLIC_GAP / 2))

# Check that areas formerly open between ribs are now a continuous sheet.
for x, y in ((15.0, 45.0), (110.0, 45.0), (30.0, 56.0), (95.0, 20.0)):
    assert carrier.is_inside((x, y, CARRIER_BASE_THICKNESS / 2)), (x, y)
# The corners are closed on the wall side, but clear the modelled head
# envelope on the acrylic side. Verify the actual heads before assembly.
for x, y in CORNER_SCREWS:
    assert carrier.is_inside((x, y, CARRIER_BASE_THICKNESS / 2)), (x, y)
    clearance_z = (CARRIER_THICKNESS + ACRYLIC_GAP
                   - CORNER_HEAD_PROJECTION - CORNER_HEAD_Z_CLEARANCE)
    head_envelope = Pos(x, y, clearance_z) * Cylinder(
        CORNER_HEAD_DIAMETER / 2, CORNER_HEAD_PROJECTION + CORNER_HEAD_Z_CLEARANCE,
        align=(Align.CENTER, Align.CENTER, Align.MIN))
    assert volume_of(carrier.intersect(head_envelope)) < 1e-5, (x, y)

individual_studs = []
for x, y in STUD_CENTERS:
    with BuildPart() as one_stud:
        with Locations((x, y, WALL_PLATE_THICKNESS)):
            Cylinder(STUD_DIAMETER / 2, STUD_HEIGHT,
                     align=(Align.CENTER, Align.CENTER, Align.MIN))
        with Locations((x, y, WALL_PLATE_THICKNESS + STUD_HEIGHT)):
            Cone(STUD_DIAMETER / 2, STUD_HEAD_DIAMETER / 2, HEAD_TAPER_HEIGHT,
                 align=(Align.CENTER, Align.CENTER, Align.MIN))
        with Locations((x, y, WALL_PLATE_THICKNESS + STUD_HEIGHT + HEAD_TAPER_HEIGHT)):
            Cylinder(STUD_HEAD_DIAMETER / 2, HEAD_TOP_THICKNESS,
                     align=(Align.CENTER, Align.CENTER, Align.MIN))
    individual_studs.append(printable_orientation(one_stud.part))

# Fixed wall plate; rotate the carrier to each orientation. Approach the wall
# through the wide entries and lower the carrier until the studs reach the
# narrow, closed ends. Also check that gravity cannot slide past those ends.
for angle in (0, 90, 180, 270):
    cx, cy = BOARD_CENTER
    turn = Pos(cx, cy, 0) * Rot(0, 0, angle) * Pos(-cx, -cy, 0)
    for height in (0, 1, 2, 3, 4, 5, 6, 8):
        moved = Pos(0, SLOT_TRAVEL, CARRIER_BOTTOM + height) * turn * print_carrier
        overlap = volume_of(print_wall_plate.intersect(moved))
        assert overlap < 1e-5, ("approach", angle, height, overlap)
    for distance in (0, SLOT_TRAVEL / 2, SLOT_TRAVEL):
        moved = Pos(0, distance, CARRIER_BOTTOM) * turn * print_carrier
        overlap = volume_of(print_wall_plate.intersect(moved))
        assert overlap < 1e-5, ("slide", angle, distance, overlap)
    below_stop = Pos(0, -1.0, CARRIER_BOTTOM) * turn * print_carrier
    for stud in individual_studs:
        assert volume_of(stud.intersect(below_stop)) > 0.01, ("gravity stop", angle)
    pulled_away = Pos(0, 0, CARRIER_BOTTOM + 1.5) * turn * print_carrier
    for stud in individual_studs:
        assert volume_of(stud.intersect(pulled_away)) > 0.01, ("head retention", angle)

# Rectangular envelopes for the major apertures in the original STEP, in XY.
cutouts = {
    "UART0-OUT": (19.5, 71.0, 32.5, 77.7),
    "UART1-OUT": (39.5, 71.0, 52.5, 77.7),
    "I2C-OUT": (59.5, 71.0, 72.5, 77.7),
    "wireless lower": (98.3, 56.3, 117.9, 58.4),
    "wireless upper": (98.0, 58.4, 118.0, 82.5),
    "S1/S0": (70.95, 20.9, 75.95, 25.9),
    "speaker": (33.05, 3.6, 42.05, 11.05),
    "UART0-IN": (106.5, 1.5, 122.5, 9.4),
    "left button notch": (-1.0, 19.27, 8.34, 39.78),
}
for label, (x1, y1, x2, y2) in cutouts.items():
    aperture = Pos((x1+x2)/2, (y1+y2)/2, 0) * Box(
        x2-x1, y2-y1, CARRIER_THICKNESS + ACRYLIC_GAP,
        align=(Align.CENTER, Align.CENTER, Align.MIN))
    overlap = volume_of(carrier.intersect(aperture))
    assert overlap < 1e-5, (label, overlap)
    # Also check the wall plate and studs behind the opening in each orientation.
    aperture_full_depth = Pos((x1+x2)/2, (y1+y2)/2, 0) * Box(
        x2-x1, y2-y1, HEAD_TOP + 0.1, align=(Align.CENTER, Align.CENTER, Align.MIN))
    for angle in (0, 90, 180, 270):
        cx, cy = BOARD_CENTER
        unturn = Pos(cx, cy, 0) * Rot(0, 0, -angle) * Pos(-cx, -cy, 0)
        overlap = volume_of((unturn * wall_plate).intersect(aperture_full_depth))
        assert overlap < 1e-5, ("wall opening", label, angle, overlap)
print("Passed: supplier assembly clearance, valid parts, holes, all four paths and cutouts.")

## 6. Export

Run after the checks. The STLs are separate and already oriented for printing:
carrier flat with 1 mm bosses up; wall plate flat with studs up. The mount-only
and complete-device STEP assemblies are positioning previews, **not** single
printable objects.

In [ ]:
from collections import Counter
import struct

export_dir = CAD_DIR / "exports"
export_dir.mkdir(exist_ok=True)
export_stl(print_carrier, export_dir / "acrylic_carrier.stl", tolerance=0.05, angular_tolerance=0.12)
export_step(print_carrier, export_dir / "acrylic_carrier.step")
export_stl(print_wall_plate, export_dir / "wall_plate.stl", tolerance=0.05, angular_tolerance=0.12)
export_step(print_wall_plate, export_dir / "wall_plate.step")
export_step(
    Compound(label="Wall mount", children=[print_wall_plate, assembled_carrier]),
    export_dir / "mount_assembly.step",
)
export_step(
    Compound(label="Complete display and wall mount",
             children=[print_wall_plate, assembled_carrier, *assembled_original]),
    export_dir / "complete_display_assembly.step",
)

def check_stl(path):
    # Check binary STL edge pairing and signed volume after export.
    data = path.read_bytes()
    count = struct.unpack_from("<I", data, 80)[0]
    assert len(data) == 84 + 50 * count and count > 0
    edges = Counter()
    volume = 0.0
    for index in range(count):
        values = struct.unpack_from("<12fH", data, 84 + 50 * index)
        a, b, c = (values[3:6], values[6:9], values[9:12])
        vertices = [tuple(round(v, 5) for v in p) for p in (a, b, c)]
        for i, j in ((0, 1), (1, 2), (2, 0)):
            edges[tuple(sorted((vertices[i], vertices[j])))] += 1
        cross = (b[1]*c[2] - b[2]*c[1],
                 b[2]*c[0] - b[0]*c[2],
                 b[0]*c[1] - b[1]*c[0])
        volume += sum(a[i] * cross[i] for i in range(3)) / 6
    assert all(uses == 2 for uses in edges.values()), f"{path.name}: non-manifold mesh"
    assert volume > 0, f"{path.name}: invalid signed volume"
    print(f"{path.name}: {count} triangles, closed mesh, {volume / 1000:.1f} cm³")

check_stl(export_dir / "acrylic_carrier.stl")
check_stl(export_dir / "wall_plate.stl")
print(f"Exported to {export_dir}")